<a href="https://colab.research.google.com/github/Sakith-N/Statistical-Learning-e22252/blob/assignment-7b/Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
1.

In [2]:
from scipy.stats import beta
import plotly.graph_objects as go
import numpy as np

x = np.linspace(0, 1, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 1, 1), name="Alpha=1, Beta=1 (Uniform)"))
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 2, 8), name="Alpha=2, Beta=8 (Right-skewed)"))
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 8, 2), name="Alpha=8, Beta=2 (Left-skewed)"))
fig.update_layout(title="Beta Distribution Configurations", xaxis_title="theta", yaxis_title="Density")
fig.show()

2.

*   Isolated trial likelihood:$L(y_k|\theta) = \theta^{y_k}(1-\theta)^{1-y_k}$

*   Joint likelihood function for running history vector $y^{(k)}$:$L(y^{(k)}|\theta) = \prod_{i=1}^k \theta^{y_i}(1-\theta)^{1-y_i} = \theta^{\sum y_i}(1-\theta)^{k - \sum y_i}$



3. Applying Bayes' theorem to find the step-$k$ posterior distribution:
$f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto L(y_k|\theta) \cdot f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)})$

  $f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto \left[\theta^{y_k}(1-\theta)^{1-y_k}\right] \cdot \left[\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}\right]$

  $f_{\Theta|Y^{(k)}}(\theta|y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1}(1-\theta)^{(\beta_{k-1} + 1 - y_k) - 1}$

Because this expression matches the functional form of a Beta distribution, the posterior remains conjugate within the same family, defined by the parameters:$\alpha_k = \alpha_{k-1} + y_k$
$\beta_k = \beta_{k-1} + 1 - y_k$

The resulting analytical Posterior Mean is:$\mathbb{E}[\Theta|Y^{(k)}=y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$

4.

*   A click ($y_k=1$) increases $\alpha_k$ while keeping $\beta_k$ constant. shift the distribution peak rightward.
*   A non-click ($y_k=0$) increments $\beta_k$. shift the peak leftward.



5. Running Posterior Mean: $\hat{\theta}_{Bayes}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$

  Running Maximum A Posteriori (MAP):
$\hat{\theta}_{MAP}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k, \beta_k > 1)$

In [3]:
np.random.seed(42)
theta_true = 0.35
n_impressions = 100

alpha, beta_val = 1, 1
mean_track, map_track = [alpha / (alpha + beta_val)], [0.5] # Standard uniform mode initialized at 0.5

for k in range(1, n_impressions + 1):
    yk = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha += yk
    beta_val += (1 - yk)

    mean_track.append(alpha / (alpha + beta_val))
    map_track.append((alpha - 1) / (alpha + beta_val - 2) if (alpha + beta_val > 2) else 0.5)

fig = go.Figure()
fig.add_trace(go.Scatter(y=mean_track, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(y=map_track, mode='lines', name='MAP'))
fig.add_trace(go.Scatter(x=[0, n_impressions], y=[theta_true, theta_true], mode='lines', name='True CTR', line=dict(dash='dash')))
fig.update_layout(title="Beta Binomial Sequential Tracking", xaxis_title="Impressions", yaxis_title="CTR Estimate")
fig.show()